<a href="https://colab.research.google.com/github/gabrielmprata/anatel_banda_larga_fixa/blob/main/PREP_banda_larga_fixa_pwr_bi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img loading="lazy" src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/python/python-original.svg" width="40" height="40"/> <img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/pandas/pandas-original-wordmark.svg" width="40" height="40"/>   <img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

---

>
**Dev**: Gabriel Prata
>
**Data**: 13/05/2026
>
**Última modificação**: 13/05/2026
>
**Contexto**: *Preparar os Dados abertos de banda larga fixa.*
>
---

Pré-processamento dos dados, para serem utilizados em um painel elaborado em Power BI.
>
<img loading="lazy" src="https://img.icons8.com/?size=100&id=3sGOUDo9nJ4k&format=png&color=000000" width="40" height="40"/>

>
![Badge em Desenvolvimento](http://img.shields.io/static/v1?label=STATUS&message=EM%20DESENVOLVIMENTO&color=GREEN&style=for-the-badge)

![Badge versao](http://img.shields.io/static/v1?label=Ver.&message=v3.0&color=red&style=for-the-badge&logo=github)

#**<font color=#4c60d6 size="6"> Import libraries**

In [2]:
# Importação de pacotes
import pandas as pd
import numpy as np
import missingno as ms # para tratamento de missings
import datetime
import calendar
import re # expressão regulares
import unicodedata

#bibliotecas para visualização de dados
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#compactar
from shutil import copyfileobj
import bz2
# Data e hora
from datetime import datetime, date, time

# Configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

#**<font color=#85d338 size="6"> 1. Coleta de dados**

As informações de acessos de **Banda Larga Fixa**, estão no sítio de dados abertos do governo federal, no link abaixo:
>
https://dados.gov.br/dados/conjuntos-dados/acessos---banda-larga-fixa

>
É disponibilizado um arquivo ZIP com todos os anos disponíveis.
>
Para esse projeto o arquivo foi coletado no dia **13/05/2026**.
>
Selecionamos os seguintes arquivos de **2025**, para tratar:
>
```
Acessos_Banda_Larga_Fixa_2025.zip
```
>
E o arquivo de **2024**, para analisar a evolução:
>
```
Acessos_Banda_Larga_Fixa_2024.zip
```

In [5]:
# importando dataset

# Conjunto de dados com os acessos de 2025
df_acessos_bl_2025 = pd.read_csv("Acessos_Banda_Larga_Fixa_2025.csv.bz2", compression='bz2', delimiter=';')

# Conjunto de dados com os acessos de 2024
df_acessos_bl_2024 = pd.read_csv("Acessos_Banda_Larga_Fixa_2024.csv.bz2", compression='bz2', delimiter=';')


No dataset de 2024, só temos interesse no mês de **DEZEMBRO**.
>
Vamos criar um Dataframe selecionado, e depois concatenar com o Dataframe de 2025.

In [9]:
df_acessos_bl_2024 = df_acessos_bl_2024[(df_acessos_bl_2024['Mês'] == 12)].copy()

Utilizei a função "concat" para juntar os DataFranes, em apenas um.

In [10]:
df_acessos_bl = pd.concat([df_acessos_bl_2025, df_acessos_bl_2024],sort=False, ignore_index=True)



---



#**<font color=#85d338 size="6"> 2. Análise de Dados Inicial**

###**<font color=#85d338> 2.1. Estatísticas Descritivas**

Compreende a organização, o resumo e, descrever os dados, que podem ser expressos em tabelas e gráficos.
>
Veremos a seguir alguns comandos para exibir algumas estatísticas descritivas.


In [11]:
#	Quantidade de atributos e instâncias (linhas/colunas)
df_acessos_bl.shape

(8819006, 16)

<font color=#85d338> Data frame com 16 atributos(features) e cerca de 8.8 milhões de tuplas.
>


---





In [12]:
# Exibir os 5 primeiros registros
df_acessos_bl.head(5)

,Ano,Mês,Grupo Econômico,Empresa,CNPJ,Porte da Prestadora,UF,Município,Código IBGE Município,Faixa de Velocidade,Velocidade,Tecnologia,Meio de Acesso,Tipo de Pessoa,Tipo de Produto,Acessos
0,2025,12,OUTROS,ARION SERVICES,20059578000120,Pequeno Porte,SP,Guarulhos,3518800,2Mbps a 12Mbps,"12,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,30
1,2025,12,OUTROS,ARION SERVICES,20059578000120,Pequeno Porte,MG,Confins,3117876,> 34Mbps,"100,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,23
2,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Amparo,3501905,12Mbps a 34Mbps,"20,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,3
3,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Borebi,3507456,> 34Mbps,"307,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,1
4,2025,12,TELEFONICA,VIVO,2558157000162,Grande Porte,SP,Iacri,3519204,512kbps a 2Mbps,"2,000000",ETHERNET,Cabo Metálico,Pessoa Jurídica,OUTROS,1




---



In [13]:
# Mostra diversas informações do Dataframe em um único comando, e exibir o uso de memória
df_acessos_bl.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 16 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   Ano                    int64 
 1   Mês                    int64 
 2   Grupo Econômico        object
 3   Empresa                object
 4   CNPJ                   int64 
 5   Porte da Prestadora    object
 6   UF                     object
 7   Município              object
 8   Código IBGE Município  int64 
 9   Faixa de Velocidade    object
 10  Velocidade             object
 11  Tecnologia             object
 12  Meio de Acesso         object
 13  Tipo de Pessoa         object
 14  Tipo de Produto        object
 15  Acessos                int64 
dtypes: int64(5), object(11)
memory usage: 5.8 GB


<font color=#85d338> Data frame com cerca de **6 gigas de memória**.


---

In [14]:
# Quantidade de valores únicos
df_acessos_bl.nunique()

,0
Ano,2
Mês,12
Grupo Econômico,24
Empresa,18830
CNPJ,11868
Porte da Prestadora,2
UF,27
Município,5297
Código IBGE Município,5570
Faixa de Velocidade,5




---



In [15]:
# Quantidade de NaN/Missing/Nulls no dataframe
df_acessos_bl.isnull().sum()

,0
Ano,0
Mês,0
Grupo Econômico,205
Empresa,205
CNPJ,0
Porte da Prestadora,205
UF,0
Município,0
Código IBGE Município,0
Faixa de Velocidade,0




---



###**<font color=#85d338> 2.2. Distribuição dos atributos**

>Nessa etapa, iremos verificar a distribuição dos principais atributos. Para ver se existe a necessidade de tomar alguma ação de transformações na etapa de preparação de dados.


---

In [21]:
df_acessos_bl.describe().round(2)

,Ano,Mês,CNPJ,Código IBGE Município,Acessos
count,8819006.00,8819006.00,8.819006e+06,8819006.00,8819006.00
mean,2024.92,6.95,2.688351e+13,3439790.39,79.74
std,0.26,3.61,2.450851e+13,906882.88,1579.75
min,2024.00,1.00,5.727400e+10,1100015.00,1.00
25%,2025.00,4.00,6.346446e+12,2919306.00,1.00
50%,2025.00,7.00,1.794785e+13,3505708.00,3.00
75%,2025.00,10.00,4.043254e+13,4203907.00,13.00
max,2025.00,12.00,9.755455e+13,5300108.00,666982.00




---



#**<font color=#85d338 size="6"> 3. Pré-Processamento de dados**

Após coletar e analisar os dados na etapa anterior, agora é o momento
de limpar, transformar e apresentar melhor os dados.
>
Assim poderemos obter, na próxima etapa, os melhores resultados possíveis nos algoritmos de
Machine Learning, ou simplesmente apresentar dados mais confiáveis para os clientes em soluções de
business intelligence.


---

###**<font color=#85d338> 3.1. Limpeza**

De forma resumida, a limpeza consiste na verificação da consistência das informações, correção de possíveis erros de preenchimento ou eliminação de valores desconhecidos, redundantes ou não pertencentes ao domínio.



####**<font color=#85d338> 3.1.1 Padronização de dados**

Dentro da programação, possuímos alguns padrões de escrita para nomes de variáveis, funções, classes e assim por diante.
>
Esses padrões de escrita são chamados de estilos de case.
>
Existem diversos tipos de case, nesse projeto iremos utilizar:
>
**Snake Case (snake_case)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).
>
**Remover acentos (remover_acentos)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).

In [23]:
def remover_acentos(texto):
    # Decompõe caracteres acentuados (ex: á -> a + ´)
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # Remove os caracteres de acentuação (intervalo unicode)
    return re.sub(r'[\u0300-\u036f]', '', texto_normalizado)

df_acessos_bl.columns = [remover_acentos(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

Index(['Ano', 'Mes', 'Grupo Economico', 'Empresa', 'CNPJ',
       'Porte da Prestadora', 'UF', 'Municipio', 'Codigo IBGE Municipio',
       'Faixa de Velocidade', 'Velocidade', 'Tecnologia', 'Meio de Acesso',
       'Tipo de Pessoa', 'Tipo de Produto', 'Acessos'],
      dtype='object')

Aplicar **Snake Case**

In [25]:
# Criar uma função para aplicar o snake_case
def snake_case(string):
    string = re.sub(" +", " ", string)   # substitui múltiplos espaços por um espaço
    string = re.sub(" ", "_", string)    # substitui espaço por _
    return string.lower() # transforma em minuscula

df_acessos_bl.columns = [snake_case(column) for column in df_acessos_bl.columns]
df_acessos_bl.columns

Index(['ano', 'mes', 'grupo_economico', 'empresa', 'cnpj',
       'porte_da_prestadora', 'uf', 'municipio', 'codigo_ibge_municipio',
       'faixa_de_velocidade', 'velocidade', 'tecnologia', 'meio_de_acesso',
       'tipo_de_pessoa', 'tipo_de_produto', 'acessos'],
      dtype='object')

In [26]:
df_acessos_bl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 16 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   ano                    int64 
 1   mes                    int64 
 2   grupo_economico        object
 3   empresa                object
 4   cnpj                   int64 
 5   porte_da_prestadora    object
 6   uf                     object
 7   municipio              object
 8   codigo_ibge_municipio  int64 
 9   faixa_de_velocidade    object
 10  velocidade             object
 11  tecnologia             object
 12  meio_de_acesso         object
 13  tipo_de_pessoa         object
 14  tipo_de_produto        object
 15  acessos                int64 
dtypes: int64(5), object(11)
memory usage: 1.1+ GB


####**<font color=#85d338> 3.1.2 Redundâncias**

Vamos eliminar as colunas que não iremos utilizar em nossas analises.
>
A ideia é ter um dataframe mais leve, e com pouco espaço em disco.

In [27]:
df_acessos_bl.drop([
				            'cnpj','tecnologia'
                    ,'tipo_de_pessoa'
                    ,'tipo_de_produto'
                    ], axis=1, inplace= True)

In [29]:
df_acessos_bl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8819006 entries, 0 to 8819005
Data columns (total 12 columns):
 #   Column                 Dtype 
---  ------                 ----- 
 0   ano                    int64 
 1   mes                    int64 
 2   grupo_economico        object
 3   empresa                object
 4   porte_da_prestadora    object
 5   uf                     object
 6   municipio              object
 7   codigo_ibge_municipio  int64 
 8   faixa_de_velocidade    object
 9   velocidade             object
 10  meio_de_acesso         object
 11  acessos                int64 
dtypes: int64(4), object(8)
memory usage: 807.4+ MB


####**<font color=#85d338> 3.1.2 Tratamento de Missings**

Como o DataFrame tem muitos atributos, e muitos deles possuem valores nulos, vou utlizar um método para mostrar apenas os valores nulos e o percentual.
>
1) Converter a função isnull() em um pd.series e depois transformar em DF, assim conseguimos filtrar quais atributos são nulos.
>

In [30]:
#1)
# cria um pd.series
dfnull = df_acessos_bl.isnull().sum()

# Converte series em dataframe
dfnull = (dfnull.to_frame(name="QTD"))

tot = len(df_acessos_bl) #Total de registros no dataset

#Criano o atributo perc, para saber o percentual de registros nulo do atributo
dfnull["perc"] = ((dfnull['QTD']/tot)*100).round(2)

#Mostrar apenas os atributos com valores nulos, ordenando para o com mais linhas nulas
dfnull.query('QTD > 0').sort_values(by='perc', ascending=False)

,QTD,perc
grupo_economico,205,0.0
empresa,205,0.0
porte_da_prestadora,205,0.0


Como todos os atributos são metricas, preenchido com a quatidade de ocorrencias registradas, iremos preencher todos com zero.

In [ ]:
df_ocorrencias['feminicidio'] = df_ocorrencias['feminicidio'].fillna(0)
df_ocorrencias['tentativa_feminicidio'] = df_ocorrencias['tentativa_feminicidio'].fillna(0)
df_ocorrencias['roubo_bicicleta'] = df_ocorrencias['roubo_bicicleta'].fillna(0)
df_ocorrencias['furto_bicicleta'] = df_ocorrencias['furto_bicicleta'].fillna(0)
df_ocorrencias['apf'] = df_ocorrencias['apf'].fillna(0)
df_ocorrencias['posse_drogas'] = df_ocorrencias['posse_drogas'].fillna(0)
df_ocorrencias['trafico_drogas'] = df_ocorrencias['trafico_drogas'].fillna(0)
df_ocorrencias['apreensao_drogas_sem_autor'] = df_ocorrencias['apreensao_drogas_sem_autor'].fillna(0)
df_ocorrencias['cmp'] = df_ocorrencias['cmp'].fillna(0)
df_ocorrencias['aaapai'] = df_ocorrencias['aaapai'].fillna(0)
df_ocorrencias['cmba'] = df_ocorrencias['cmba'].fillna(0)
df_ocorrencias['roubo_cx_eletronico'] = df_ocorrencias['roubo_cx_eletronico'].fillna(0)

####**<font color=#85d338> 3.1.3 Outliers**

In [ ]:
import plotly.express as px

fig = px.box(df_ocorrencias, x='ano', y="total_furtos")
fig.update_xaxes(type="category", title=None) # se o type for date, vai respeitar o intervalo


fig.show()

In [ ]:
df_ocorrencias[['cisp','ano','mes','munic',
                'furto_veiculos','furto_transeunte','furto_coletivo',
                'furto_celular','furto_bicicleta','outros_furtos']][(df_ocorrencias['total_furtos'] == 1665)]

,cisp,ano,mes,munic,furto_veiculos,furto_transeunte,furto_coletivo,furto_celular,furto_bicicleta,outros_furtos
31709,16,2022,9,Rio de Janeiro,21,103,59,992,5.0,485


In [ ]:
df_ocorrencias[['cisp','ano','mes','munic','total_furtos',
                'furto_veiculos','furto_transeunte','furto_coletivo',
                'furto_celular','furto_bicicleta','outros_furtos']][(df_ocorrencias['cisp'] == 16) &
                                                                    (df_ocorrencias['ano'] == 2022)]

,cisp,ano,mes,munic,total_furtos,furto_veiculos,furto_transeunte,furto_coletivo,furto_celular,furto_bicicleta,outros_furtos
30613,16,2022,1,Rio de Janeiro,646,21,42,48,175,6.0,354
30750,16,2022,2,Rio de Janeiro,658,30,71,56,235,13.0,253
30887,16,2022,3,Rio de Janeiro,773,46,56,71,263,11.0,326
31024,16,2022,4,Rio de Janeiro,774,27,57,71,281,7.0,331
31161,16,2022,5,Rio de Janeiro,566,25,39,70,164,7.0,261
31298,16,2022,6,Rio de Janeiro,493,17,28,50,164,9.0,225
31435,16,2022,7,Rio de Janeiro,495,23,29,46,152,8.0,237
31572,16,2022,8,Rio de Janeiro,683,18,37,62,262,7.0,297
31709,16,2022,9,Rio de Janeiro,1665,21,103,59,992,5.0,485
31846,16,2022,10,Rio de Janeiro,585,34,36,47,210,7.0,251


In [ ]:
import plotly.express as px

fig = px.box(df_ocorrencias, x='ano', y="total_roubos")
fig.update_xaxes(type="category", title=None) # se o type for date, vai respeitar o intervalo


fig.show()

In [ ]:
df_ocorrencias[['cisp','ano','mes','munic',
                'roubo_transeunte','roubo_celular','roubo_em_coletivo',
                'roubo_rua','roubo_veiculo','total_roubos']][(df_ocorrencias['total_roubos'] == 1160)]

,cisp,ano,mes,munic,roubo_transeunte,roubo_celular,roubo_em_coletivo,roubo_rua,roubo_veiculo,total_roubos
22966,64,2017,5,São João de Meriti,482,122,107,711,284,1160


In [ ]:
df_ocorrencias[['cisp','ano','mes','munic',
                'roubo_transeunte','roubo_celular','roubo_em_coletivo',
                'roubo_rua','roubo_veiculo','total_roubos']][(df_ocorrencias['cisp'] == 64) &
                                                             (df_ocorrencias['ano'] == 2017)]

,cisp,ano,mes,munic,roubo_transeunte,roubo_celular,roubo_em_coletivo,roubo_rua,roubo_veiculo,total_roubos
22414,64,2017,1,São João de Meriti,174,44,47,265,296,643
22552,64,2017,2,São João de Meriti,28,12,6,46,273,365
22690,64,2017,3,São João de Meriti,53,18,32,103,317,507
22828,64,2017,4,São João de Meriti,302,134,80,516,276,962
22966,64,2017,5,São João de Meriti,482,122,107,711,284,1160
23104,64,2017,6,São João de Meriti,413,82,64,559,272,997
23242,64,2017,7,São João de Meriti,317,65,60,442,350,966
23380,64,2017,8,São João de Meriti,386,84,40,510,312,980
23518,64,2017,9,São João de Meriti,286,82,43,411,271,836
23656,64,2017,10,São João de Meriti,315,109,48,472,248,882


####**<font color=#85d338> 3.1.4 Padronização de dados**

Dentro da programação, possuímos alguns padrões de escrita para nomes de variáveis, funções, classes e assim por diante.
>
Esses padrões de escrita são chamados de estilos de case.
>
Existem diversos tipos de case, nesse projeto iremos utilizar:
>
**Snake Case (snake_case)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).
>
**Remover acentos (remover_acentos)**: Nesse estilo, todas as letras são minúsculas e as palavras são separadas por um underscore(_).

Nesse projeto somente o conjunto de dados de CISP, precisa de padronização de dados.

><font color=#85d338>**Conjunto de dados CISP**

In [ ]:
df_cisp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147 entries, 0 to 146
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   CISP                 147 non-null    int64 
 1   Unidade Territorial  147 non-null    object
 2   Município            147 non-null    object
 3   Região de Governo    147 non-null    object
dtypes: int64(1), object(3)
memory usage: 4.7+ KB


In [ ]:
def remover_acentos(texto):
    # Decompõe caracteres acentuados (ex: á -> a + ´)
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # Remove os caracteres de acentuação (intervalo unicode)
    return re.sub(r'[\u0300-\u036f]', '', texto_normalizado)

df_cisp.columns = [remover_acentos(column) for column in df_cisp.columns]
df_cisp.columns

Index(['CISP', 'Unidade Territorial', 'Municipio', 'Regiao de Governo'], dtype='object')

In [ ]:
# Criar uma função para aplicar o snake_case
def snake_case(string):
    string = re.sub(" +", " ", string)   # substitui múltiplos espaços por um espaço
    string = re.sub(" ", "_", string)    # substitui espaço por _
    return string.lower() # transforma em minuscula

df_cisp.columns = [snake_case(column) for column in df_cisp.columns]
df_cisp.columns

Index(['cisp', 'unidade_territorial', 'municipio', 'regiao_de_governo'], dtype='object')

Substituir o separador dos valores de virgula para ponto e virgula

In [ ]:
df_cisp['unidade_territorial'] = df_cisp['unidade_territorial'].str.replace(',',';', regex=True)



---



>Atributo Região

In [ ]:
df_ocorrencias.regiao.value_counts(normalize=True)

,proportion
regiao,
Interior,0.487538
Capital,0.300075
Baixada Fluminense,0.138152
Grande NiterÃÂÃÂÃÂÃÂ³i,0.069947
Grande Niterói,0.004288


In [ ]:
df_ocorrencias.loc[df_ocorrencias.regiao=='Grande NiterÃÂÃÂÃÂÃÂ³i','regiao']='Grande Niterói'

In [ ]:
df_ocorrencias.regiao.value_counts(normalize=True)

,proportion
regiao,
Interior,0.487538
Capital,0.300075
Baixada Fluminense,0.138152
Grande Niterói,0.074235


####**<font color=#85d338> 3.1.5 Duplicidade de registros**

In [ ]:
# Verificando se existe duplicidade na dimensão
df_cisp.duplicated(subset='cisp', keep='first').sum()

np.int64(10)

In [ ]:
# Exibindo os registros duplicados
duplicidade = df_cisp['cisp'].duplicated(keep='first')
df_cisp[duplicidade]

,cisp,unidade_territorial,municipio,regiao_de_governo
88,96,Paty do Alferes e Avelar,Paty do Alferes,Centro-Sul Fluminense
101,100,Quatis; Falcão e Ribeirão de São Joaquim,Quatis,Médio Paraíba
110,140,Varre-Sai,Varre-Sai,Noroeste Fluminense
112,143,São José de Ubá,São José de Ubá,Noroeste Fluminense
115,148,Cardoso Moreira e São Joaquim,Cardoso Moreira,Norte Fluminense
121,130,Quissamã,Quissamã,Norte Fluminense
124,136,Santo Antônio de Pádua; Campelo; Paraoquena; M...,Santo Antônio de Pádua,Noroeste Fluminense
132,154,Macuco,Macuco,Serrana
144,108,Comendador Levy Gasparian e Afonso Arinos,Comendador Levy Gasparian,Centro-Sul Fluminense
145,108,Três Rios e Bemposta,Três Rios,Centro-Sul Fluminense


In [ ]:
# Detalhando um registro duplicado como exemplo
df_cisp.query('cisp == 108' )

,cisp,unidade_territorial,municipio,regiao_de_governo
143,108,Areal,Areal,Centro-Sul Fluminense
144,108,Comendador Levy Gasparian e Afonso Arinos,Comendador Levy Gasparian,Centro-Sul Fluminense
145,108,Três Rios e Bemposta,Três Rios,Centro-Sul Fluminense


In [ ]:
# Removendo as duplicidades
df_cisp = df_cisp[~duplicidade]

In [ ]:
# Validando se ainda existe duplicidade na dimensão
df_cisp.duplicated(subset='cisp', keep='first').sum()

np.int64(0)



---



###**<font color=#4c60d6> 3.2 Criação de recursos**

Também conhecida como ***feature engineering***, a criação de recursos consiste em criar, a partir dos atributos originais, um conjunto de atributos que capture informações importantes.

####**<font color=#4c60d6> 3.2.1 Construção de recuros**

><font color=#4c60d6>**Atributo data e hora**

Criar um atributo com o ano e o mês para facilitar as analises gráficas.

In [ ]:
df_ocorrencias['ano_mes'] = df_ocorrencias['ano'].map(str) + '-' + df_ocorrencias['mes'].map(str)
df_ocorrencias['ano_mes'] = pd.to_datetime(df_ocorrencias['ano_mes'])
df_ocorrencias['ano_mes'] = df_ocorrencias['ano_mes'].dt.strftime('%Y-%m')

><font color=#4c60d6>**Atributo mês em caracter**

In [ ]:
# Mes em caracter
df_ocorrencias['mes_char'] = df_ocorrencias['mes']

dic_mes = {
1: 'Janeiro',
2: 'Fevereiro',
3: 'Março',
4: 'Abril',
5: 'Maio',
6: 'Junho',
7: 'Julho',
8: 'Agosto',
9: 'Setembro',
10: 'Outubro',
11: 'Novembro',
12: 'Dezembro'}

# Fazer o replace nos atributos conforme o dicionario
df_ocorrencias = df_ocorrencias.replace({
    'mes_char' : dic_mes
})

><font color=#4c60d6>**Atributo Regiões integradas de segurança pública (RISP)**

In [ ]:
# RISP corresponência
df_ocorrencias['risp_desc'] = df_ocorrencias['risp']

dic_risp = {
1: 'Capital (Zona Sul, Centro e parte da Norte)',
2: 'Capital (Zona Oeste e parte da Norte)',
3: 'Baixada Fluminense',
4: 'Grande Niterói e Região dos Lagos',
5: 'Sul Fluminense',
6: 'Norte Fluminense e Noroeste',
7: 'Região Serrana'
}

# Fazer o replace nos atributos conforme o dicionario
df_ocorrencias = df_ocorrencias.replace({
    'risp_desc' : dic_risp
})

> Atributo com o total de policiais mortos em serviço

In [ ]:
df_ocorrencias['pol_mortos_serv'] = df_ocorrencias['pol_militares_mortos_serv']+df_ocorrencias['pol_civis_mortos_serv']

####**<font color=#4c60d6> 3.2.2 Enriquecimento**

Existem outros atributos que iremos relacionar com outro dataset.
>
Seria como um modelo relacional, porém nesse caso, iremos adicionar no dataset principal(Fato), os atributos das dimensões.

><font color=#4c60d6>**Circunscrições Integradas de Segurança Pública  (CISP)**

In [ ]:
df_ocorrencias = pd.merge(df_ocorrencias, df_cisp, left_on=['cisp'], right_on=['cisp'], how='left')

###**<font color=#4c60d6> 3.3 Redução da dimensionalidade**

O dataset conta com informações de 2003 a 2026(parcial).
>
Vou construir um dataset para mostrar a evolução ao longo dos anos.

><font color=#4c60d6>**Por ano e mês**

In [ ]:
df_hist_anual = df_ocorrencias[['ano','mes','mes_char','total','letalidade_violenta','hom_doloso','lesao_corp_morte','latrocinio','cvli','hom_por_interv_policial','feminicidio','pol_militares_mortos_serv','pol_civis_mortos_serv','total_roubos','total_furtos','roubo_carga','roubo_veiculo','furto_veiculos','roubo_rua','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','pol_mortos_serv','recuperacao_veiculos','furto_transeunte','furto_coletivo','furto_celular']][(df_ocorrencias['ano'] <= 2025)]
df_hist_anual = df_hist_anual.groupby(['ano','mes','mes_char']).sum(['total','letalidade_violenta','hom_doloso','lesao_corp_morte','latrocinio','cvli','hom_por_interv_policial','feminicidio','pol_militares_mortos_serv','pol_civis_mortos_serv','total_roubos','total_furtos','roubo_carga','roubo_veiculo','furto_veiculos','roubo_rua','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','pol_mortos_serv','recuperacao_veiculos','furto_transeunte','furto_coletivo','furto_celular']).reset_index()


><font color=#4c60d6>**Comparação Por ano(todos)**

In [ ]:
df_hs_compara = df_hist_anual.groupby('ano')[['total','letalidade_violenta','hom_doloso', 'total_roubos', 'roubo_veiculo','furto_veiculos', 'roubo_rua', 'roubo_carga','latrocinio','feminicidio','apreensao_drogas','pol_mortos_serv', 'estelionato', 'hom_por_interv_policial','lesao_corp_morte','apf','aaapai','roubo_transeunte','roubo_celular','recuperacao_veiculos','furto_transeunte','furto_coletivo','furto_celular']].apply(lambda x: x.sum()).reset_index()

In [ ]:
# Criar dataframe para comparar a variação ano contra ano em percentual e absoluto

# Criar métrica com o valor do ano anterior
df_hs_compara['total_ant'] = df_hs_compara.total.shift(1)
df_hs_compara['letalidade_violenta_ant'] = df_hs_compara.letalidade_violenta.shift(1)
df_hs_compara['hom_doloso_ant'] = df_hs_compara.hom_doloso.shift(1)
df_hs_compara['roubos_ant'] = df_hs_compara.total_roubos.shift(1)
df_hs_compara['roubo_veiculo_ant'] = df_hs_compara.roubo_veiculo.shift(1)
df_hs_compara['roubo_rua_ant'] = df_hs_compara.roubo_rua.shift(1)
df_hs_compara['roubo_carga_ant'] = df_hs_compara.roubo_carga.shift(1)
df_hs_compara['latrocinio_ant'] = df_hs_compara.latrocinio.shift(1)
df_hs_compara['feminicidio_ant'] = df_hs_compara.feminicidio.shift(1)
df_hs_compara['estelionato_ant'] = df_hs_compara.estelionato.shift(1)
df_hs_compara['apreensao_drogas_ant'] = df_hs_compara.apreensao_drogas.shift(1)
df_hs_compara['pol_mortos_serv_ant'] = df_hs_compara.pol_mortos_serv.shift(1)
df_hs_compara['hom_por_interv_policial_ant'] = df_hs_compara.hom_por_interv_policial.shift(1)
df_hs_compara['lesao_corp_morte_ant'] = df_hs_compara.lesao_corp_morte.shift(1)
df_hs_compara['apf_ant'] = df_hs_compara.apf.shift(1)
df_hs_compara['aaapai_ant'] = df_hs_compara.aaapai.shift(1)
df_hs_compara['roubo_transeunte_ant'] = df_hs_compara.roubo_transeunte.shift(1)
df_hs_compara['roubo_celular_ant'] = df_hs_compara.roubo_celular.shift(1)

df_hs_compara['furto_veiculos_ant'] = df_hs_compara.furto_veiculos.shift(1)
df_hs_compara['recuperacao_veiculos_ant'] = df_hs_compara.recuperacao_veiculos.shift(1)

# Delta
df_hs_compara['dif_total'] = (df_hs_compara['total']-df_hs_compara['total_ant'])
df_hs_compara['dif_letalidade_violenta'] = (df_hs_compara['letalidade_violenta']-df_hs_compara['letalidade_violenta_ant'])
df_hs_compara['dif_hom_doloso'] = (df_hs_compara['hom_doloso']-df_hs_compara['hom_doloso_ant'])
df_hs_compara['dif_roubos'] = (df_hs_compara['total_roubos']-df_hs_compara['roubos_ant'])
df_hs_compara['dif_roubo_veiculo'] = (df_hs_compara['roubo_veiculo']-df_hs_compara['roubo_veiculo_ant'])
df_hs_compara['dif_roubo_rua'] = (df_hs_compara['roubo_rua']-df_hs_compara['roubo_rua_ant'])
df_hs_compara['dif_roubo_carga'] = (df_hs_compara['roubo_carga']-df_hs_compara['roubo_carga_ant'])
df_hs_compara['dif_latrocinio'] = (df_hs_compara['latrocinio']-df_hs_compara['latrocinio_ant'])
df_hs_compara['dif_feminicidio'] = (df_hs_compara['feminicidio']-df_hs_compara['feminicidio_ant'])
df_hs_compara['dif_estelionato'] = (df_hs_compara['estelionato']-df_hs_compara['estelionato_ant'])
df_hs_compara['dif_apreensao_drogas'] = (df_hs_compara['apreensao_drogas']-df_hs_compara['apreensao_drogas_ant'])
df_hs_compara['dif_pol_mortos_serv'] = (df_hs_compara['pol_mortos_serv']-df_hs_compara['pol_mortos_serv_ant'])
df_hs_compara['dif_hom_por_interv_policial'] = (df_hs_compara['hom_por_interv_policial']-df_hs_compara['hom_por_interv_policial_ant'])
df_hs_compara['dif_lesao'] = (df_hs_compara['lesao_corp_morte']-df_hs_compara['lesao_corp_morte_ant'])
df_hs_compara['dif_apf'] = (df_hs_compara['apf']-df_hs_compara['apf_ant'])
df_hs_compara['dif_aaapai'] = (df_hs_compara['aaapai']-df_hs_compara['aaapai_ant'])
df_hs_compara['dif_rtranseunte'] = (df_hs_compara['roubo_transeunte']-df_hs_compara['roubo_transeunte_ant'])
df_hs_compara['dif_rcelular'] = (df_hs_compara['roubo_celular']-df_hs_compara['roubo_celular_ant'])
df_hs_compara['dif_furto_veiculos'] = (df_hs_compara['furto_veiculos']-df_hs_compara['furto_veiculos_ant'])
df_hs_compara['dif_recuperacao_veiculos'] = (df_hs_compara['recuperacao_veiculos']-df_hs_compara['recuperacao_veiculos_ant'])

# Variação em percentual
df_hs_compara['var_total'] = (((df_hs_compara['total']/df_hs_compara['total_ant'])*100)-100).round(1)
df_hs_compara['var_letalidade_violenta'] = (((df_hs_compara['letalidade_violenta']/df_hs_compara['letalidade_violenta_ant'])*100)-100).round(1)
df_hs_compara['var_hom_doloso'] = (((df_hs_compara['hom_doloso']/df_hs_compara['hom_doloso_ant'])*100)-100).round(1)
df_hs_compara['var_roubos'] = (((df_hs_compara['total_roubos']/df_hs_compara['roubos_ant'])*100)-100).round(1)
df_hs_compara['var_roubo_veiculo'] = (((df_hs_compara['roubo_veiculo']/df_hs_compara['roubo_veiculo_ant'])*100)-100).round(1)
df_hs_compara['var_roubo_rua'] = (((df_hs_compara['roubo_rua']/df_hs_compara['roubo_rua_ant'])*100)-100).round(1)
df_hs_compara['var_roubo_carga'] = (((df_hs_compara['roubo_carga']/df_hs_compara['roubo_carga_ant'])*100)-100).round(1)
df_hs_compara['var_latrocinio'] = (((df_hs_compara['latrocinio']/df_hs_compara['latrocinio_ant'])*100)-100).round(1)
df_hs_compara['var_feminicidio'] = (((df_hs_compara['feminicidio']/df_hs_compara['feminicidio_ant'])*100)-100).round(1)
df_hs_compara['var_estelionato'] = (((df_hs_compara['estelionato']/df_hs_compara['estelionato_ant'])*100)-100).round(1)
df_hs_compara['var_apreensao_drogas'] = (((df_hs_compara['apreensao_drogas']/df_hs_compara['apreensao_drogas_ant'])*100)-100).round(1)
df_hs_compara['var_pol_mortos_serv'] = (((df_hs_compara['pol_mortos_serv']/df_hs_compara['pol_mortos_serv_ant'])*100)-100).round(1)
df_hs_compara['var_hom_por_interv_policial'] = (((df_hs_compara['hom_por_interv_policial']/df_hs_compara['hom_por_interv_policial_ant'])*100)-100).round(1)
df_hs_compara['var_lesao'] = (((df_hs_compara['lesao_corp_morte']/df_hs_compara['lesao_corp_morte_ant'])*100)-100).round(1)
df_hs_compara['var_apf'] = (((df_hs_compara['apf']/df_hs_compara['apf_ant'])*100)-100).round(1)
df_hs_compara['var_aaapai'] = (((df_hs_compara['aaapai']/df_hs_compara['aaapai_ant'])*100)-100).round(1)
df_hs_compara['var_rtranseunte'] = (((df_hs_compara['roubo_transeunte']/df_hs_compara['roubo_transeunte_ant'])*100)-100).round(1)
df_hs_compara['var_rcelular'] = (((df_hs_compara['roubo_celular']/df_hs_compara['roubo_celular_ant'])*100)-100).round(1)

df_hs_compara['var_furto_veiculos'] = (((df_hs_compara['furto_veiculos']/df_hs_compara['furto_veiculos_ant'])*100)-100).round(1)
df_hs_compara['var_recup_veiculos'] = (((df_hs_compara['recuperacao_veiculos']/df_hs_compara['recuperacao_veiculos_ant'])*100)-100).round(1)


In [ ]:
# tratando NaN
df_hs_compara.fillna(0, inplace=True)

><font color=#4c60d6>**Anuário**

In [ ]:
df_anuario = df_ocorrencias[(df_ocorrencias['ano'] == 2025)].copy().reset_index()

In [ ]:
#criar um atributo com a quantidade de dias de um ano
df_anuario['data'] = pd.to_datetime(df_anuario['ano_mes'])
df_anuario['dias_no_ano'] = df_anuario['data'].dt.year.map(lambda x: 366 if calendar.isleap(x) else 365)

# Ajustar campo do código IBGE para 6 dígitos
df_anuario["mcirc"] = df_anuario["mcirc"].map(str).str[:6]


><font color=#4c60d6>**Por região**

In [ ]:
df_regiao_cisp = df_ocorrencias[['ano','risp','risp_desc','cisp','mcirc','municipio','letalidade_violenta', 'hom_doloso', 'latrocinio', 'hom_por_interv_policial', 'lesao_corp_morte','pol_mortos_serv','total_roubos', 'roubo_veiculo', 'furto_veiculos', 'roubo_rua','roubo_carga','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','furto_transeunte','furto_coletivo','furto_celular']][(df_ocorrencias['ano'] == 2025)]
df_regiao_cisp = df_regiao_cisp.groupby(['ano','risp','risp_desc','cisp','mcirc','municipio']).sum(['letalidade_violenta', 'hom_doloso', 'latrocinio', 'hom_por_interv_policial', 'lesao_corp_morte','pol_mortos_serv','total_roubos', 'roubo_veiculo', 'furto_veiculos', 'roubo_rua','roubo_carga','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','furto_transeunte','furto_coletivo','furto_celular']).reset_index()



---



><font color=#4c60d6>**Por município com taxa por habitante**

In [ ]:
df_anuario.groupby(['mcirc','munic'])['letalidade_violenta'].sum().reset_index()

,mcirc,munic,letalidade_violenta
0,330010,Angra dos Reis,38
1,330020,Araruama,41
2,330023,Armação dos Búzios,17
3,330025,Arraial do Cabo,8
4,330030,Barra do Pirai,16
...,...,...,...
77,999999,Miguel Pereira;Paty dos Alferes,16
78,999999,Natividade;Varre-Sai,10
79,999999,Porto Real;Quatis,11
80,999999,Quissamã;Carapebus,8


In [ ]:
df_municipio_taxa = df_anuario[['ano','mcirc','munic','total','letalidade_violenta','hom_doloso','lesao_corp_morte','latrocinio','cvli','hom_por_interv_policial','feminicidio','pol_militares_mortos_serv','pol_civis_mortos_serv','total_roubos', 'furto_veiculos','roubo_carga','roubo_veiculo','roubo_rua','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','pol_mortos_serv','furto_transeunte','furto_coletivo','furto_celular']][(df_anuario['ano'] == 2025)]
df_municipio_taxa = df_municipio_taxa.groupby(['ano','mcirc','munic']).sum(['total','letalidade_violenta','hom_doloso','lesao_corp_morte','latrocinio','cvli','hom_por_interv_policial','feminicidio','pol_militares_mortos_serv','pol_civis_mortos_serv','total_roubos', 'furto_veiculos','roubo_carga','roubo_veiculo','roubo_rua','roubo_transeunte','roubo_celular','roubo_em_coletivo','estelionato','extorsao','apreensao_drogas','apf','aaapai','pol_mortos_serv','furto_transeunte','furto_coletivo','furto_celular']).reset_index()

In [ ]:
# converter para str
df_populacao["codigo_ibge"] = df_populacao["codigo_ibge"].map(str)

In [ ]:
# Merge
df_municipio_taxa = pd.merge(df_municipio_taxa, df_populacao, left_on=['mcirc'], right_on=['codigo_ibge'])

In [ ]:
# Calculo das taxas calculo (Sdelito/Spopulacao)*10000 populacao_residente
df_municipio_taxa["furto_rua"] = df_municipio_taxa["furto_celular"]+df_municipio_taxa["furto_coletivo"]+df_municipio_taxa["furto_transeunte"]

df_municipio_taxa["taxa_total"] = ((df_municipio_taxa["total"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_letal_violenta"] = ((df_municipio_taxa["letalidade_violenta"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_homicidio"] = ((df_municipio_taxa["hom_doloso"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_latrocinio"] = ((df_municipio_taxa["latrocinio"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_lesao_corp_morte"] = ((df_municipio_taxa["lesao_corp_morte"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_hom_interv_pol"] = ((df_municipio_taxa["hom_por_interv_policial"]/df_municipio_taxa["populacao_residente"])*10000).round(2)

df_municipio_taxa["taxa_total_roubos"] = ((df_municipio_taxa["total_roubos"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_roubo_rua"] = ((df_municipio_taxa["roubo_rua"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_roubo_veiculo"] = ((df_municipio_taxa["roubo_veiculo"]/df_municipio_taxa["populacao_residente"])*10000).round(2)

df_municipio_taxa["taxa_furto_celular"] = ((df_municipio_taxa["furto_celular"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_furto_coletivo"] = ((df_municipio_taxa["furto_coletivo"]/df_municipio_taxa["populacao_residente"])*10000).round(2)
df_municipio_taxa["taxa_furto_transeunte"] = ((df_municipio_taxa["furto_transeunte"]/df_municipio_taxa["populacao_residente"])*10000).round(2)

df_municipio_taxa["taxa_furto_rua"] = ((df_municipio_taxa["furto_rua"]/df_municipio_taxa["populacao_residente"])*10000).round(2)






><font color=#4c60d6>**historico taxa por habitante**

In [ ]:
df_hs_taxa = df_hs_compara.groupby('ano')[['total','letalidade_violenta','hom_doloso']].apply(lambda x: x.sum()).reset_index()

In [ ]:
# Merge
df_hs_taxa = pd.merge(df_hs_taxa, df_hs_estima_pop, left_on=['ano'], right_on=['ano'])

In [ ]:
# Calculo das taxas calculo (Sdelito/Spopulacao)*10000 populacao_residente
df_hs_taxa["taxa_total"] = ((df_hs_taxa["total"]/df_hs_taxa["População_residente"])*10000).round(2)
df_hs_taxa["taxa_letalidade_violenta"] = ((df_hs_taxa["letalidade_violenta"]/df_hs_taxa["População_residente"])*10000).round(2)
df_hs_taxa["taxa_hom_doloso"] = ((df_hs_taxa["hom_doloso"]/df_hs_taxa["População_residente"])*10000).round(2)
df_hs_taxa["taxa_letalidade_violenta_100"] = ((df_hs_taxa["letalidade_violenta"]/df_hs_taxa["População_residente"])*100000).round(2)
df_hs_taxa["taxa_hom_doloso_100"] = ((df_hs_taxa["hom_doloso"]/df_hs_taxa["População_residente"])*100000).round(2)

# Criar métrica com o valor do ano anterior
df_hs_taxa['taxa_total_ant'] = df_hs_taxa.taxa_total.shift(1)
df_hs_taxa['taxa_letalidade_violenta_ant'] = df_hs_taxa.taxa_letalidade_violenta.shift(1)
df_hs_taxa['taxa_hom_doloso_ant'] = df_hs_taxa.taxa_hom_doloso.shift(1)
df_hs_taxa['taxa_letalidade_violenta_100_ant'] = df_hs_taxa.taxa_letalidade_violenta_100.shift(1)
df_hs_taxa['taxa_hom_doloso_100_ant'] = df_hs_taxa.taxa_hom_doloso_100.shift(1)

# Delta
df_hs_taxa['dif_taxa_total'] = (df_hs_taxa['taxa_total']-df_hs_taxa['taxa_total_ant'])
df_hs_taxa['dif_taxa_letalidade_violenta'] = (df_hs_taxa['taxa_letalidade_violenta']-df_hs_taxa['taxa_letalidade_violenta_ant'])
df_hs_taxa['dif_taxa_hom_doloso'] = (df_hs_taxa['taxa_hom_doloso']-df_hs_taxa['taxa_hom_doloso_ant'])

# Variação em percentual
df_hs_taxa['var_taxa_total'] = (((df_hs_taxa['taxa_total']/df_hs_taxa['taxa_total_ant'])*100)-100).round(2)
df_hs_taxa['var_taxa_letalidade_violenta'] = (((df_hs_taxa['taxa_letalidade_violenta']/df_hs_taxa['taxa_letalidade_violenta_ant'])*100)-100).round(2)
df_hs_taxa['var_taxa_hom_doloso'] = (((df_hs_taxa['taxa_hom_doloso']/df_hs_taxa['taxa_hom_doloso_ant'])*100)-100).round(2)

# tratando NaN
df_hs_taxa.fillna(0, inplace=True)

><font color=#4c60d6>**Frota**

In [ ]:
# Selecionar apenas a quantidade do ultimo mes do ano
df_frota = df_frota[(df_frota['mes'] == 12)].copy()

###**<font color=#4c60d6> 3.4 Export**

Agora iremos exportar o dataset em CSV, para usarmos no Power BI

In [ ]:
# Exportar para csv
df_anuario.to_csv('df_anuario.csv', sep='|', encoding="Latin 1", index=False)

df_cisp.to_csv('df_cisp.csv', sep='|', encoding="Latin 1", index=False)

df_hist_anual.to_csv('df_hist_anual.csv', index=False)
df_hs_compara.to_csv('df_hs_compara.csv', sep=';', index=False)
df_regiao_cisp.to_csv('df_regiao_cisp.csv', sep='|', index=False)
df_municipio_taxa.to_csv('df_municipio_taxa.csv', index=False)
df_hs_taxa.to_csv('df_hs_taxa.csv', index=False)
df_frota.to_csv('df_frota.csv', index=False)
df_hs_armas.to_csv('df_hs_armas.csv', sep=';', index=False)

# compactar arquivo com nivel de compressão máxima
with open('df_anuario.csv', 'rb') as input:
    with bz2.BZ2File('df_anuario.csv.bz2', 'wb', compresslevel=9) as output:
        copyfileobj(input, output)